### **RAG for Langchain**



In [36]:
!pip -q install langchain langchain-ollama langchain-community \
    langchain-huggingface langchain-text-splitters \
    faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 82.0 MB/s eta 0:00:00


In [38]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 38 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (2,529 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [39]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [40]:
import subprocess
import time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama started")

Ollama started


In [41]:
!ollama pull llama3.2

In [42]:
!ollama run llama3.2 "What is machine learning?"

Machine learning (ML) is a subset of artificial intelligence (AI) that enab
enables computers to learn from data without being explicitly programmed. I
It involves training algorithms to make predictions, classify data, or take
take actions based on patterns and relationships learned from large dataset
datasets.

Machine learning works by feeding data into a system, which then uses stati
statistical models, such as neural networks or decision trees, to identify 
patterns and make predictions. The algorithm learns from the data and impro
improves its performance over time, allowing it to make more accurate predi
predictions and decisions.

There are several types of machine learning:

1. **Supervised learning**: The algorithm is trained on labeled data, where
where the correct output is already known. The goal is to learn a mapping b
between input data and output labels.
2. **Unsupervised learning**: The algorithm is trained on unlabeled data, a
and it must find patterns or structure in

In [43]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

response = llm.invoke(
    "Explain artificial intelligence in two sentences."
)

print(response.content)

Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making, using algorithms and data to make predictions and improve performance over time. AI systems can be designed to mimic human cognition, enabling them to understand and interact with the world in a more intelligent and autonomous way, with applications in areas like robotics, natural language processing, and image recognition.


In [44]:
import os

os.makedirs("documents", exist_ok=True)

text = """
Artificial intelligence is a field of computer science concerned
with creating systems that can perform tasks that normally require
human intelligence.

Machine learning is a subset of artificial intelligence in which
models learn patterns from data.

Deep learning is a subset of machine learning that uses neural
networks with multiple layers.

Retrieval Augmented Generation, or RAG, combines information
retrieval with a generative language model. The retrieval system
finds relevant information from an external knowledge base, and
the language model uses that information to generate an answer.

Medical imaging uses techniques such as X-ray, CT, MRI, and
ultrasound to obtain images of structures inside the human body.
"""

with open("documents/sample.txt", "w") as f:
    f.write(text)

print("Document created.")

Document created.


In [45]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("documents/sample.txt")

documents = loader.load()

print("Documents:", len(documents))

Documents: 1


In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 4


In [47]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings ready.


In [48]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS database created.")

FAISS database created.


In [57]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

vectorstore.save_local("faiss_index")

print("Embeddings saved to FAISS database.")

Embeddings saved to FAISS database.


In [49]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [51]:
results = retriever.invoke(
    "What is deep learning?"
)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Deep learning is a subset of machine learning that uses neural
networks with multiple layers.

--- Result 2 ---
Artificial intelligence is a field of computer science concerned
with creating systems that can perform tasks that normally require
human intelligence.

Machine learning is a subset of artificial intelligence in which
models learn patterns from data.


In [52]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Use ONLY the information in the context below
to answer the question.

If the answer is not contained in the context,
say that you don't know based on the provided documents.

Context:
{context}

Question:
{question}

Answer:
""")

In [53]:
def ask_rag(question):

    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # Build prompt
    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    # Send to Llama
    response = llm.invoke(messages)

    return response.content

Testing RAG

In [54]:
answer = ask_rag(
    "What is deep learning?"
)

print(answer)

Deep learning is a subset of machine learning that uses neural networks with multiple layers.


In [55]:
answer = ask_rag(
    "What is retrieval augmented generation?"
)

print(answer)

Retrieval Augmented Generation (RAG) combines information retrieval with a generative language model.


In [56]:
answer = ask_rag(
    "Who was the first president of Pakistan?"
)

print(answer)

I don't know based on the provided documents.


In [58]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS loaded


In [59]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

creating tool for langchain

In [60]:
from langchain.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the knowledge base for information relevant
    to the user's question.
    """

    documents = retriever.invoke(query)

    if not documents:
        return "No relevant information was found."

    results = []

    for doc in documents:
        results.append(doc.page_content)

    return "\n\n".join(results)

In [61]:
print(search_knowledge_base.name)
print(search_knowledge_base.description)

search_knowledge_base
Search the knowledge base for information relevant
to the user's question.


Creating AGENT

In [62]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_knowledge_base],
    system_prompt="""
You are a question-answering agent.

You have access to a knowledge-base search tool.

When the user asks a question that can be answered
using the knowledge base, use the search_knowledge_base
tool to retrieve relevant information.

Use the retrieved information to answer the user.

Do not invent information that is not supported by
the retrieved context.

If the knowledge base does not contain enough
information to answer the question, say that the
information was not found in the knowledge base.
"""
)

print("Agent created")

Agent created


In [64]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is deep learning?"
            }
        ]
    }
)

In [65]:
for message in result["messages"]:
    print("\n---", message.type, "---")
    print(message.content)


--- human ---
What is deep learning?

--- ai ---


--- tool ---
Deep learning is a subset of machine learning that uses neural
networks with multiple layers.

Artificial intelligence is a field of computer science concerned
with creating systems that can perform tasks that normally require
human intelligence.

Machine learning is a subset of artificial intelligence in which
models learn patterns from data.

Retrieval Augmented Generation, or RAG, combines information
retrieval with a generative language model. The retrieval system
finds relevant information from an external knowledge base, and
the language model uses that information to generate an answer.

--- ai ---
Deep learning is a subset of machine learning that uses neural networks with multiple layers to analyze and interpret data. It is a type of artificial intelligence that enables computers to learn from large amounts of data, such as images, speech, and text, and make predictions or take actions without being explicitly pr

In [66]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the capital of France?"
            }
        ]
    }
)

print(result["messages"][-1].content)

The capital of France is Paris.


## **RAG for LamaIndex**




In [67]:
!pip -q install llama-index llama-index-llms-ollama \
    llama-index-embeddings-huggingface \
    llama-index-vector-stores-faiss \
    faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 12.0 MB/s eta 0:00:00


In [72]:
!pkill ollama

In [73]:
import subprocess
import time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama restarted")

Ollama restarted


In [74]:
!ollama run llama3.2 "Explain artificial intelligence in two sentences."

Artificial intelligence (AI) refers to the development of computer systems 
that can perform tasks that typically require human intelligence, such as l
learning, problem-solving, and decision-making, using algorithms and data t
to improve their performance over time. AI can be applied in various fields
fields, including robotics, natural language processing, and image recognit
recognition, and has the potential to automate many tasks, improve efficien
efficiency, and enhance human productivity.



In [76]:
from llama_index.llms.ollama import Ollama

llm = Ollama(
    model="llama3.2",
    temperature=0,
    context_window=2048,
    request_timeout=120.0
)

response = llm.complete(
    "Explain artificial intelligence in two sentences."
)

print(response)

Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making, using algorithms and data to make predictions and improve performance over time. AI systems can be designed to mimic human cognition, enabling them to understand and interact with the world in a more intelligent and autonomous way, with applications in areas like robotics, natural language processing, and image recognition.


In [77]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    "documents"
).load_data()

print(f"Number of documents: {len(documents)}")
print(documents[0].text[:500])

Number of documents: 1

Artificial intelligence is a field of computer science concerned
with creating systems that can perform tasks that normally require
human intelligence.

Machine learning is a subset of artificial intelligence in which
models learn patterns from data.

Deep learning is a subset of machine learning that uses neural
networks with multiple layers.

Retrieval Augmented Generation, or RAG, combines information
retrieval with a generative language model. The retrieval system
finds relevant information


In [78]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [79]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

In [80]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents
)

print("Vector index created.")

Vector index created.


In [81]:
query_engine = index.as_query_engine(
    similarity_top_k=3
)

Testing RAG

In [82]:
response = query_engine.query(
    "What is deep learning?"
)

print(response)

Deep learning is a subset of machine learning that uses neural networks with multiple layers to analyze and interpret data.


In [83]:
response = query_engine.query(
    "What is the capital of France?"
)

print(response)

I'm not aware of any information that relates to the capital of France in the provided context.


In [84]:
retriever = index.as_retriever(
    similarity_top_k=3
)

In [86]:
retrieved_nodes = retriever.retrieve(
    "What is deep learning?"
)

for i, node in enumerate(retrieved_nodes):
    print(f"\n===== RESULT {i+1} =====")
    print(node.text)
    print("Score:", node.score)


===== RESULT 1 =====
Artificial intelligence is a field of computer science concerned
with creating systems that can perform tasks that normally require
human intelligence.

Machine learning is a subset of artificial intelligence in which
models learn patterns from data.

Deep learning is a subset of machine learning that uses neural
networks with multiple layers.

Retrieval Augmented Generation, or RAG, combines information
retrieval with a generative language model. The retrieval system
finds relevant information from an external knowledge base, and
the language model uses that information to generate an answer.

Medical imaging uses techniques such as X-ray, CT, MRI, and
ultrasound to obtain images of structures inside the human body.
Score: 0.6693577481082447


In [87]:
response = query_engine.query(
    "What is deep learning?"
)

print("ANSWER:")
print(response)

print("\nSOURCES:")
for source in response.source_nodes:
    print("\n---")
    print(source.text)
    print("Score:", source.score)

ANSWER:
Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn patterns from data.

SOURCES:

---
Artificial intelligence is a field of computer science concerned
with creating systems that can perform tasks that normally require
human intelligence.

Machine learning is a subset of artificial intelligence in which
models learn patterns from data.

Deep learning is a subset of machine learning that uses neural
networks with multiple layers.

Retrieval Augmented Generation, or RAG, combines information
retrieval with a generative language model. The retrieval system
finds relevant information from an external knowledge base, and
the language model uses that information to generate an answer.

Medical imaging uses techniques such as X-ray, CT, MRI, and
ultrasound to obtain images of structures inside the human body.
Score: 0.6693577481082447


Creating tool for Lamaindex

In [88]:
from llama_index.core.tools import QueryEngineTool

query_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="knowledge_base_search",
    description=(
        "Search the knowledge base for information about "
        "artificial intelligence, machine learning, deep learning, "
        "RAG, and medical imaging."
    )
)

Creating Agent

In [89]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[query_tool],
    llm=llm,
    system_prompt="""
You are a question-answering agent.

You have access to a knowledge-base search tool.

When the user's question can be answered using the
knowledge base, use the knowledge_base_search tool.

Use the information returned by the tool to answer.

Do not invent information that is not supported by
the knowledge base.

If the knowledge base does not contain enough
information, clearly say that the information was
not found in the knowledge base.
"""
)

print("Agent created.")

Agent created.


In [90]:
response = await agent.run(
    "What is deep learning?"
)

print(response)

Deep learning is a type of machine learning that uses artificial neural networks, which are composed of layers of interconnected nodes or "neurons," to analyze and interpret data. These neural networks are designed to mimic the structure and function of the human brain, allowing them to learn and improve their performance on a specific task over time.

Deep learning algorithms are typically used for tasks such as:

1. Image recognition and classification
2. Natural language processing (NLP) and text analysis
3. Speech recognition and synthesis
4. Time series forecasting and prediction
5. Reinforcement learning and game playing

Deep learning models are often trained on large datasets and use a variety of techniques, such as convolutional neural networks (CNNs), recurrent neural networks (RNNs), and long short-term memory (LSTM) networks, to learn patterns and relationships in the data.

Some of the key benefits of deep learning include:

1. Ability to learn complex patterns and relatio

In [91]:
response = await agent.run(
    "What is the capital of France?"
)

print(response)

The capital of France is Paris.
